# 03 Funnel Analysis

This notebook analyzes the ecommerce funnel using the cleaned dataset and focuses only on funnel metrics and simple visualizations.

## 1. Import libraries and define file paths

We import the libraries we need and set the input and output paths for the funnel analysis files.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

INPUT_PATH = Path("..") / "outputs" / "events_cleaned.csv"
OUTPUT_DIR = Path("..") / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OVERALL_OUTPUT = OUTPUT_DIR / "funnel_overall.csv"
MONTHLY_OUTPUT = OUTPUT_DIR / "funnel_monthly.csv"
CATEGORY_OUTPUT = OUTPUT_DIR / "funnel_category.csv"
BRAND_OUTPUT = OUTPUT_DIR / "funnel_brand.csv"

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print(f"Input file: {INPUT_PATH.resolve()}")

## 2. Load the cleaned dataset

We read the cleaned dataset from the `outputs` folder.

In [ ]:
df = pd.read_csv(INPUT_PATH, parse_dates=["event_time"])
print("Dataset loaded successfully.")

## 3. Briefly inspect the cleaned dataset

This is a quick check to confirm the structure before calculating funnel metrics.

In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Event types:", sorted(df["event_type"].dropna().unique().tolist()))
df.head()

## 4. Keep only the main funnel stages

For this project, the funnel stages are `view`, `cart`, and `purchase`.

In [ ]:
funnel_stages = ["view", "cart", "purchase"]
funnel_df = df[df["event_type"].isin(funnel_stages)].copy()
print("Shape after filtering to funnel stages:", funnel_df.shape)

## 5. Define a helper function for funnel metrics

This function calculates counts and conversion rates for any grouping level.

In [ ]:
def calculate_funnel_metrics(dataframe, group_col=None):
    if group_col is None:
        counts = dataframe["event_type"].value_counts()
        result = pd.DataFrame({
            "views": [counts.get("view", 0)],
            "carts": [counts.get("cart", 0)],
            "purchases": [counts.get("purchase", 0)],
        })
    else:
        counts = (
            dataframe.groupby([group_col, "event_type"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=funnel_stages, fill_value=0)
            .reset_index()
        )
        result = counts.rename(columns={
            "view": "views",
            "cart": "carts",
            "purchase": "purchases",
        })

    result["view_to_cart_rate"] = (result["carts"] / result["views"]).where(result["views"] > 0, 0)
    result["cart_to_purchase_rate"] = (result["purchases"] / result["carts"]).where(result["carts"] > 0, 0)
    result["view_to_purchase_rate"] = (result["purchases"] / result["views"]).where(result["views"] > 0, 0)

    return result

## 6. Calculate overall funnel metrics

This gives us the main funnel counts and conversion rates for the full dataset.

In [ ]:
funnel_overall = calculate_funnel_metrics(funnel_df)
funnel_overall

## 7. Calculate monthly funnel metrics

This shows how the funnel changes over time by `event_month`.

In [ ]:
funnel_monthly = calculate_funnel_metrics(funnel_df, group_col="event_month").sort_values("event_month")
funnel_monthly.head()

## 8. Calculate category funnel metrics

This summarizes funnel performance by product category.

In [ ]:
funnel_category = calculate_funnel_metrics(funnel_df, group_col="category_code")
funnel_category = funnel_category.sort_values(["view_to_purchase_rate", "purchases"], ascending=[False, False])
funnel_category.head(10)

## 9. Calculate brand funnel metrics

This summarizes funnel performance by brand.

In [ ]:
funnel_brand = calculate_funnel_metrics(funnel_df, group_col="brand")
funnel_brand = funnel_brand.sort_values(["view_to_purchase_rate", "purchases"], ascending=[False, False])
funnel_brand.head(10)

## 10. Save funnel summary tables

These CSV files can be reused later for dashboards and reporting.

In [ ]:
funnel_overall.to_csv(OVERALL_OUTPUT, index=False)
funnel_monthly.to_csv(MONTHLY_OUTPUT, index=False)
funnel_category.to_csv(CATEGORY_OUTPUT, index=False)
funnel_brand.to_csv(BRAND_OUTPUT, index=False)

print(f"Saved: {OVERALL_OUTPUT.resolve()}")
print(f"Saved: {MONTHLY_OUTPUT.resolve()}")
print(f"Saved: {CATEGORY_OUTPUT.resolve()}")
print(f"Saved: {BRAND_OUTPUT.resolve()}")

## 11. Visualize the overall funnel counts

This bar chart shows the total number of events at each funnel stage.

In [ ]:
overall_plot = funnel_overall[["views", "carts", "purchases"]].T.reset_index()
overall_plot.columns = ["stage", "count"]

sns.barplot(data=overall_plot, x="stage", y="count", hue="stage", palette="Blues_d", legend=False)
plt.title("Overall Funnel Counts")
plt.xlabel("Funnel Stage")
plt.ylabel("Event Count")
plt.tight_layout()
plt.show()

## 12. Visualize the monthly funnel trend

This line chart shows how views, carts, and purchases change by month.

In [ ]:
monthly_plot = funnel_monthly[["event_month", "views", "carts", "purchases"]].melt(
    id_vars="event_month",
    value_vars=["views", "carts", "purchases"],
    var_name="stage",
    value_name="count",
)

sns.lineplot(data=monthly_plot, x="event_month", y="count", hue="stage", marker="o")
plt.title("Monthly Funnel Trend")
plt.xlabel("Event Month")
plt.ylabel("Event Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. Visualize the top categories by purchase conversion

To keep the chart meaningful, we only look at categories with at least 100 views.

In [ ]:
top_category_plot = (
    funnel_category[funnel_category["views"] >= 100]
    .nlargest(10, "view_to_purchase_rate")
    .copy()
)

sns.barplot(
    data=top_category_plot,
    x="view_to_purchase_rate",
    y="category_code",
    hue="category_code",
    palette="Greens_d",
    legend=False,
)
plt.title("Top Categories by View-to-Purchase Conversion")
plt.xlabel("Conversion Rate")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

## 14. Visualize the top brands by purchase conversion

To keep the chart meaningful, we only look at brands with at least 100 views.

In [ ]:
top_brand_plot = (
    funnel_brand[funnel_brand["views"] >= 100]
    .nlargest(10, "view_to_purchase_rate")
    .copy()
)

sns.barplot(
    data=top_brand_plot,
    x="view_to_purchase_rate",
    y="brand",
    hue="brand",
    palette="Oranges_d",
    legend=False,
)
plt.title("Top Brands by View-to-Purchase Conversion")
plt.xlabel("Conversion Rate")
plt.ylabel("Brand")
plt.tight_layout()
plt.show()

## 15. Print a short summary of the funnel findings

This final section gives a simple text summary of the main results from the funnel analysis.

In [ ]:
overall_row = funnel_overall.iloc[0]
best_month = funnel_monthly.sort_values("view_to_purchase_rate", ascending=False).iloc[0]
best_category = funnel_category[funnel_category["views"] >= 100].sort_values("view_to_purchase_rate", ascending=False).iloc[0]
best_brand = funnel_brand[funnel_brand["views"] >= 100].sort_values("view_to_purchase_rate", ascending=False).iloc[0]

print("Key Funnel Findings")
print("- Total views:", int(overall_row["views"]))
print("- Total carts:", int(overall_row["carts"]))
print("- Total purchases:", int(overall_row["purchases"]))
print(f"- View-to-cart conversion rate: {overall_row['view_to_cart_rate']:.2%}")
print(f"- Cart-to-purchase conversion rate: {overall_row['cart_to_purchase_rate']:.2%}")
print(f"- View-to-purchase conversion rate: {overall_row['view_to_purchase_rate']:.2%}")
print(f"- Best month by view-to-purchase conversion: {best_month['event_month']} ({best_month['view_to_purchase_rate']:.2%})")
print(f"- Top category by view-to-purchase conversion with at least 100 views: {best_category['category_code']} ({best_category['view_to_purchase_rate']:.2%})")
print(f"- Top brand by view-to-purchase conversion with at least 100 views: {best_brand['brand']} ({best_brand['view_to_purchase_rate']:.2%})")